In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as mp
import seaborn as sns
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

import warnings
warnings.filterwarnings("ignore")


from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()

from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

/kaggle/input/datasets/akshaymairal/store-item-demand-forecasting-challenge/sample_submission.csv
/kaggle/input/datasets/akshaymairal/store-item-demand-forecasting-challenge/train.csv
/kaggle/input/datasets/akshaymairal/store-item-demand-forecasting-challenge/test.csv
/kaggle/input/datasets/siddharth0231/festivals-2013-2017/festivals_base.xlsx


# 1

In [2]:
# data of sales
df = pd.read_csv("/kaggle/input/datasets/akshaymairal/store-item-demand-forecasting-challenge/train.csv")
# data of festivals
fd = pd.read_excel("/kaggle/input/datasets/siddharth0231/festivals-2013-2017/festivals_base.xlsx")

In [3]:
df

,date,store,item,sales
0,2013-01-01,1,1,13
1,2013-01-02,1,1,11
2,2013-01-03,1,1,14
3,2013-01-04,1,1,13
4,2013-01-05,1,1,10
...,...,...,...,...
912995,2017-12-27,10,50,63
912996,2017-12-28,10,50,59
912997,2017-12-29,10,50,74
912998,2017-12-30,10,50,62


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 913000 entries, 0 to 912999
Data columns (total 4 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   date    913000 non-null  object
 1   store   913000 non-null  int64 
 2   item    913000 non-null  int64 
 3   sales   913000 non-null  int64 
dtypes: int64(3), object(1)
memory usage: 27.9+ MB


changing object into data 

# 2

In [5]:
df["date"] = pd.to_datetime(df['date'])

In [6]:
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
df["day"] = df["date"].dt.day
df["weekday"] = df["date"].dt.day_name()

In [7]:
print(fd.shape)
fd.tail(13)

(190, 6)


,festival_name,festival_date,festival_type,region,impact_scale,is_regional_event
177,End_of_Season_Sale_Jul,2015-07-15,Shopping_Event,National,5,0
178,End_of_Season_Sale_Jul,2016-07-15,Shopping_Event,National,5,0
179,End_of_Season_Sale_Jul,2017-07-15,Shopping_Event,National,5,0
180,Back_to_School,2013-06-01,Shopping_Event,National,4,0
181,Back_to_School,2014-06-01,Shopping_Event,National,4,0
182,Back_to_School,2015-06-01,Shopping_Event,National,4,0
183,Back_to_School,2016-06-01,Shopping_Event,National,4,0
184,Back_to_School,2017-06-01,Shopping_Event,National,4,0
185,Big_Billion_Day,2013-10-06,Shopping_Event,National,7,0
186,Big_Billion_Day,2014-10-06,Shopping_Event,National,7,0


In [8]:
fd.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 190 entries, 0 to 189
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   festival_name      190 non-null    object
 1   festival_date      190 non-null    object
 2   festival_type      190 non-null    object
 3   region             190 non-null    object
 4   impact_scale       190 non-null    int64 
 5   is_regional_event  190 non-null    int64 
dtypes: int64(2), object(4)
memory usage: 9.0+ KB


# 3

In [9]:
fd["festival_date"] = pd.to_datetime(fd['festival_date'])

In [10]:
fd['festival_type'].value_counts()

festival_type
Major             90
Minor             75
Shopping_Event    25
Name: count, dtype: int64

In [11]:
fd['region'].value_counts()

region
National    130
North        25
East         15
South        15
West          5
Name: count, dtype: int64

In [12]:
fd['is_regional_event'].value_counts()

is_regional_event
0    130
1     60
Name: count, dtype: int64

In [13]:
fd['festival_name'].value_counts()

festival_name
Diwali                    5
Dhanteras                 5
Holi                      5
Eid_ul_Fitr               5
Eid_ul_Adha               5
Navratri                  5
Dussehra                  5
Raksha_Bandhan            5
Janmashtami               5
Ganesh_Chaturthi          5
Christmas                 5
Makar_Sankranti           5
Maha_Shivratri            5
Ram_Navami                5
Guru_Nanak_Jayanti        5
Bhai_Dooj                 5
Vasant_Panchami           5
Republic_Day              5
Independence_Day          5
New_Year                  5
Karwa_Chauth              5
Chhath_Puja               5
Lohri                     5
Baisakhi                  5
Durga_Puja                5
Pongal                    5
Onam                      5
Ugadi                     5
Gudi_Padwa                5
Bihu                      5
Teej                      5
Rath_Yatra                5
Govardhan_Puja            5
Wedding_Season_Start      5
End_of_Season_Sale_Jan    5
End_of

In [14]:
fd['impact_scale'].value_counts()

impact_scale
5     50
7     40
6     35
4     25
8     20
9     15
10     5
Name: count, dtype: int64

In [15]:
#fd["year"] = fd["festival_date"].dt.year
#fd["month"] = fd["festival_date"].dt.month
#fd["day"] = fd["festival_date"].dt.day
#fd["weekday"] = fd["festival_date"].dt.day_name()

In [16]:
fd.columns

Index(['festival_name', 'festival_date', 'festival_type', 'region',
       'impact_scale', 'is_regional_event'],
      dtype='object')

# 4 

In [17]:
big_d = pd.merge(df,fd,left_on = "date" , right_on = "festival_date" , how = "left")

# 5

In [18]:
big_d["year"] = big_d["date"].dt.year
big_d["month"] = big_d["date"].dt.month
big_d["day"] = big_d["date"].dt.day
big_d["weekday"] = big_d["date"].dt.day_name()

In [19]:
big_d.tail(16)

,date,store,item,sales,year,month,day,weekday,festival_name,festival_date,festival_type,region,impact_scale,is_regional_event
919984,2017-12-16,10,50,52,2017,12,16,Saturday,NaN,NaT,NaN,NaN,NaN,NaN
919985,2017-12-17,10,50,86,2017,12,17,Sunday,NaN,NaT,NaN,NaN,NaN,NaN
919986,2017-12-18,10,50,53,2017,12,18,Monday,NaN,NaT,NaN,NaN,NaN,NaN
919987,2017-12-19,10,50,54,2017,12,19,Tuesday,NaN,NaT,NaN,NaN,NaN,NaN
919988,2017-12-20,10,50,51,2017,12,20,Wednesday,NaN,NaT,NaN,NaN,NaN,NaN
919989,2017-12-21,10,50,63,2017,12,21,Thursday,NaN,NaT,NaN,NaN,NaN,NaN
919990,2017-12-22,10,50,75,2017,12,22,Friday,NaN,NaT,NaN,NaN,NaN,NaN
919991,2017-12-23,10,50,70,2017,12,23,Saturday,NaN,NaT,NaN,NaN,NaN,NaN
919992,2017-12-24,10,50,76,2017,12,24,Sunday,NaN,NaT,NaN,NaN,NaN,NaN
919993,2017-12-25,10,50,51,2017,12,25,Monday,Christmas,2017-12-25,Major,National,7.0,0.0


In [20]:
big_d["impact_scale"].value_counts()

impact_scale
5.0     25000
7.0     20000
6.0     17500
4.0     12500
8.0     10000
9.0      7500
10.0     2500
Name: count, dtype: int64

In [21]:
big_d["festival_name"].value_counts()

festival_name
New_Year                  2500
Lohri                     2500
Makar_Sankranti           2500
Pongal                    2500
End_of_Season_Sale_Jan    2500
Republic_Day              2500
Vasant_Panchami           2500
Maha_Shivratri            2500
Ram_Navami                2500
Holi                      2500
Ugadi                     2500
Gudi_Padwa                2500
Baisakhi                  2500
Bihu                      2500
Back_to_School            2500
Rath_Yatra                2500
End_of_Season_Sale_Jul    2500
Eid_ul_Fitr               2500
Teej                      2500
Independence_Day          2500
Raksha_Bandhan            2500
Janmashtami               2500
Ganesh_Chaturthi          2500
Onam                      2500
Navratri                  2500
Big_Billion_Day           2500
Durga_Puja                2500
Dussehra                  2500
Eid_ul_Adha               2500
Karwa_Chauth              2500
Dhanteras                 2500
Diwali                   

In [22]:
big_d.index

RangeIndex(start=0, stop=920000, step=1)

In [23]:
data_100 = big_d.head(100)
data_100

,date,store,item,sales,year,month,day,weekday,festival_name,festival_date,festival_type,region,impact_scale,is_regional_event
0,2013-01-01,1,1,13,2013,1,1,Tuesday,New_Year,2013-01-01,Minor,National,5.0,0.0
1,2013-01-02,1,1,11,2013,1,2,Wednesday,NaN,NaT,NaN,NaN,NaN,NaN
2,2013-01-03,1,1,14,2013,1,3,Thursday,NaN,NaT,NaN,NaN,NaN,NaN
3,2013-01-04,1,1,13,2013,1,4,Friday,NaN,NaT,NaN,NaN,NaN,NaN
4,2013-01-05,1,1,10,2013,1,5,Saturday,NaN,NaT,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,2013-04-05,1,1,19,2013,4,5,Friday,NaN,NaT,NaN,NaN,NaN,NaN
96,2013-04-06,1,1,23,2013,4,6,Saturday,NaN,NaT,NaN,NaN,NaN,NaN
97,2013-04-07,1,1,17,2013,4,7,Sunday,NaN,NaT,NaN,NaN,NaN,NaN
98,2013-04-08,1,1,19,2013,4,8,Monday,NaN,NaT,NaN,NaN,NaN,NaN


In [24]:
ldate = data_100["festival_date"].dropna()
ldate

0    2013-01-01
12   2013-01-13
13   2013-01-14
14   2013-01-14
20   2013-01-20
26   2013-01-26
46   2013-02-15
69   2013-03-10
78   2013-03-19
86   2013-03-27
Name: festival_date, dtype: datetime64[ns]

In [25]:
loco = ldate.loc[0]
loco

Timestamp('2013-01-01 00:00:00')

In [26]:
off = list(range(7,-3))

In [27]:

data_100['festival_name'] = le.fit_transform(data_100["festival_name"])
data_100['weekday'] = le.fit_transform(data_100["weekday"])
data_100['festival_type'] = le.fit_transform(data_100["festival_type"])
data_100['region'] = le.fit_transform(data_100["region"])


In [28]:
data_100

,date,store,item,sales,year,month,day,weekday,festival_name,festival_date,festival_type,region,impact_scale,is_regional_event
0,2013-01-01,1,1,13,2013,1,1,5,5,2013-01-01,1,0,5.0,0.0
1,2013-01-02,1,1,11,2013,1,2,6,10,NaT,3,3,NaN,NaN
2,2013-01-03,1,1,14,2013,1,3,4,10,NaT,3,3,NaN,NaN
3,2013-01-04,1,1,13,2013,1,4,0,10,NaT,3,3,NaN,NaN
4,2013-01-05,1,1,10,2013,1,5,2,10,NaT,3,3,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,2013-04-05,1,1,19,2013,4,5,0,10,NaT,3,3,NaN,NaN
96,2013-04-06,1,1,23,2013,4,6,2,10,NaT,3,3,NaN,NaN
97,2013-04-07,1,1,17,2013,4,7,3,10,NaT,3,3,NaN,NaN
98,2013-04-08,1,1,19,2013,4,8,1,10,NaT,3,3,NaN,NaN


In [29]:
big_d['festival_name'] = le.fit_transform(big_d["festival_name"])
big_d['weekday'] = le.fit_transform(big_d["weekday"])
big_d['festival_type'] = le.fit_transform(big_d["festival_type"])
big_d['region'] = le.fit_transform(big_d["region"])

In [30]:
big_d.columns

Index(['date', 'store', 'item', 'sales', 'year', 'month', 'day', 'weekday',
       'festival_name', 'festival_date', 'festival_type', 'region',
       'impact_scale', 'is_regional_event'],
      dtype='object')

In [31]:
import pandas as pd

# Step 1: Create an empty list to hold all expanded rows
expanded_list = []

# Step 2: Loop through each festival date in ldate
for festival_date in ldate:
    
    # Step 3: Generate the 11-day window (7 days before to 3 days after)
    dates = pd.date_range(start=festival_date - pd.Timedelta(days=7), periods=11, freq='D')
    
    # Step 4: Generate the offset numbers (-7 to +3)
    offsets = list(range(-7, 4))
    
    # Step 5: Create a mini DataFrame for this festival
    mini_df = pd.DataFrame({
        'festival_date': dates,
        'days_to_next_festival': offsets,
        'is_festival_day': [1 if offset == 0 else 0 for offset in offsets],
        'pre_festival_week': [1 if -7 <= offset <= 0 else 0 for offset in offsets]
    })
    
    # Step 6: Add this mini DataFrame to our list
    expanded_list.append(mini_df)

# Step 7: Combine all mini DataFrames into one big DataFrame
fd_expanded = pd.concat(expanded_list, ignore_index=True)

# Step 8: Check the result
print(f"Created {len(fd_expanded)} rows")
print(fd_expanded.head(15))

Created 110 rows
   festival_date  days_to_next_festival  is_festival_day  pre_festival_week
0     2012-12-25                     -7                0                  1
1     2012-12-26                     -6                0                  1
2     2012-12-27                     -5                0                  1
3     2012-12-28                     -4                0                  1
4     2012-12-29                     -3                0                  1
5     2012-12-30                     -2                0                  1
6     2012-12-31                     -1                0                  1
7     2013-01-01                      0                1                  1
8     2013-01-02                      1                0                  0
9     2013-01-03                      2                0                  0
10    2013-01-04                      3                0                  0
11    2013-01-06                     -7                0               

In [32]:
final_df = pd.merge(big_d, fd_expanded, left_on='date', right_on='festival_date', how='left')

data sales shift 

In [33]:
#final_df["sales_lag_1"] = final_df["sales"]
#final_df["sales_lag_1"] = final_df["sales_lag_1"].shift(periods = 1)

In [34]:
# CORRECT way to create lag features
final_df['sales_lag_1'] = final_df.groupby(['store', 'item'])['sales'].shift(1)
final_df['sales_lag_7'] = final_df.groupby(['store', 'item'])['sales'].shift(7)

In [35]:
final_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 939500 entries, 0 to 939499
Data columns (total 20 columns):
 #   Column                 Non-Null Count   Dtype         
---  ------                 --------------   -----         
 0   date                   939500 non-null  datetime64[ns]
 1   store                  939500 non-null  int64         
 2   item                   939500 non-null  int64         
 3   sales                  939500 non-null  int64         
 4   year                   939500 non-null  int32         
 5   month                  939500 non-null  int32         
 6   day                    939500 non-null  int32         
 7   weekday                939500 non-null  int64         
 8   festival_name          939500 non-null  int64         
 9   festival_date_x        100000 non-null  datetime64[ns]
 10  festival_type          939500 non-null  int64         
 11  region                 939500 non-null  int64         
 12  impact_scale           100000 non-null  floa

In [36]:
from sklearn.model_selection import train_test_split

# dividing data around year

In [37]:
X_data = final_df[final_df["year"] < 2017]
Y_data = final_df[final_df["year"] == 2017]

# x_data (before 2017)

In [38]:
X = X_data.drop(["sales" , "festival_date_x" ,"festival_date_y", "date"], axis = 1)
y = X_data["sales"]

In [39]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Y_data (2017)

In [40]:
Y_ka_X = Y_data.drop(["sales" , "festival_date_x" ,"festival_date_y", "date"], axis = 1)
Y_ka_y = Y_data["sales"]

In [41]:
from sklearn.metrics import accuracy_score

In [42]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    n_estimators=100,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1
)

In [43]:
rf.fit(X_train , y_train)

RandomForestRegressor(max_features='sqrt', n_jobs=-1, random_state=42)

In [44]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score , root_mean_squared_error

# Predict
predictions = rf.predict(Y_ka_X )

# Calculate metrics
mae = mean_absolute_error(Y_ka_y, predictions)

r2 = r2_score(Y_ka_y, predictions)

print(f"MAE: {mae:.2f}")

print(f"R² Score: {r2:.2f}")

MAE: 7.18
R² Score: 0.91


1st -> 
MAE: 16.40
R² Score: 0.47



2nd -> MAE: 2.39
R² Score: 0.49

3rd -> MAE: 2.63
R² Score: 0.42

4th -> MAE: 9.83
R² Score: 0.82 (max_depth = 10)

MAE: 8.39
R² Score: 0.88  (at max_depth = 15)



MAE: 7.63
R² Score: 0.90


MAE: 10.11
R² Score: 0.81

MAE: 7.18
R² Score: 0.91